In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu, randint, loguniform

from lifelines import KaplanMeierFitter
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (make_scorer,roc_auc_score,roc_curve, precision_score, fbeta_score, f1_score, recall_score,
confusion_matrix, classification_report, accuracy_score, ConfusionMatrixDisplay)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings("ignore") # suppress noncritical warnings to keep output clean

# Read Heart Failure Dataset

In [11]:
df = pd.read_csv("..\data\heart_failure_clinical_records_dataset.csv")
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


We prepare dataset for machine learning modeling

# Define X and y

In [14]:
drop_cols = ["time", "DEATH_EVENT"] # we droped time to avoid data leakage because follow up time is part of the survival outcomes
X = df.drop(columns = drop_cols)
y = df["DEATH_EVENT"]
X.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0


In [15]:
y.head()

0    1
1    1
2    1
3    1
4    1
Name: DEATH_EVENT, dtype: int64

# Split Dataset

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [18]:
# We extract numeric and categorical features
numeric_cols = ["age", "creatinine_phosphokinase", "ejection_fraction", "platelets", "serum_creatinine", "serum_sodium"]
categorical_cols = ["anaemia", "diabetes", "high_blood_pressure", "sex","smoking"]

# Logistic Regression Model

In [20]:
#We preprocess the columns in a transformer
preprocessor = ColumnTransformer(transformers =[("num", StandardScaler(), numeric_cols),("cat", "passthrough", categorical_cols)], remainder="drop")

In [21]:
# We buid a pipeline
log_pipeline = Pipeline(steps=[("preprocessor", preprocessor),
                        ("classifier",LogisticRegression(max_iter=1000, solver="lbfgs", C = 0.1, class_weight="balanced", random_state= 42))])

In [22]:
# We fit the model
log_pipeline.fit(X_train, y_train);

We check the model performance on training set

In [24]:
y_train_pred = log_pipeline.predict(X_train)
print(classification_report(y_train, y_train_pred, target_names=["no_death_event","death_event"]))

                precision    recall  f1-score   support

no_death_event       0.88      0.78      0.83       162
   death_event       0.63      0.77      0.69        77

      accuracy                           0.78       239
     macro avg       0.75      0.78      0.76       239
  weighted avg       0.80      0.78      0.78       239



We check the model performance on test set

In [26]:
y_test_pred = log_pipeline.predict(X_test)
print(classification_report(y_test, y_test_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.84      0.78      0.81        41
   death_event       0.59      0.68      0.63        19

      accuracy                           0.75        60
     macro avg       0.72      0.73      0.72        60
  weighted avg       0.76      0.75      0.75        60



In [27]:
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

[[32  9]
 [ 6 13]]


The logistic regression model correctly identified 68% of the actual death-event cases in the test set. There were 19 actual death-event cases, and the model detected approximately 13 of them. The precision for the death-event class was also 0.59, meaning that among patients predicted as death-event cases, about 59% truly experienced a death event. Although the model shows moderate ability to detect mortality cases, it still misses some of the actual death-event patients, which is an important limitation in a healthcare risk prediction setting.

## Hyperparameter Tuning

We tune the hyperparameters and employ a cross validation for possible improve performance.

In [31]:
para_grid = {"classifier__C": [0.01, 0.02, 0.03,0.04,0.05], "classifier__penalty": ["l2"], 
             "classifier__solver": ["lbfgs"], "classifier__class_weight": ["balanced"] }

In [32]:
cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state =42)
score = make_scorer(recall_score,labels =[1], average="macro")
random_search = RandomizedSearchCV(estimator=log_pipeline,
                          param_distributions= para_grid,
                          n_iter = 5,
                          cv = cv,
                          scoring=score,
                          refit = True,
                          random_state=42,
                          n_jobs=-1)

In [33]:
random_search.fit(X_train, y_train);

In [34]:
print("Best parameters:", random_search.best_params_)

Best parameters: {'classifier__solver': 'lbfgs', 'classifier__penalty': 'l2', 'classifier__class_weight': 'balanced', 'classifier__C': 0.03}


In [35]:
# We choose the best model
best_logistic_model = random_search.best_estimator_

In [36]:
# We predict using the best model
y_train_pred = best_logistic_model.predict(X_train)

In [37]:
# We check performance on training set
print(classification_report(y_train, y_train_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.87      0.80      0.83       162
   death_event       0.63      0.74      0.68        77

      accuracy                           0.78       239
     macro avg       0.75      0.77      0.76       239
  weighted avg       0.79      0.78      0.78       239



In [38]:
# We check performance on test set
y_test_pred = best_logistic_model.predict(X_test)

In [39]:
# We check the performance
print(classification_report(y_test, y_test_pred, target_names = ["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.86      0.78      0.82        41
   death_event       0.61      0.74      0.67        19

      accuracy                           0.77        60
     macro avg       0.74      0.76      0.74        60
  weighted avg       0.78      0.77      0.77        60



After applying cross-validation and hyperparameter tuning, the logistic regression model achieved a recall of 0.74 for the death-event class on the test set. This means the model correctly detected approximately 74% of the patients who actually experienced a death event. The model also achieved a precision of 0.61 for the death-event class, meaning that among patients predicted as death-event cases, 61% were truly death-event patients. Overall, the tuned model shows moderate ability to identify mortality-risk cases, with improved sensitivity to death events, although some false positives and false negatives remain. The similarity between the training and test performance suggests that the model generalizes reasonably well and does not show strong evidence of overfitting.

# Random Forest Model

Next we fit the RF model.

In [43]:
# This part is not really neccessary since tree based models are not sensitive to scaling, but I've reatined it since I'm learning to maintain a professional workflow.

rf_preprocessor = ColumnTransformer(transformers = [("num", "passthrough",numeric_cols ), ("cat","passthrough", categorical_cols)], remainder = "drop")

In [44]:
# We build a RF model pipeline
rf_pipeline = Pipeline(steps = [("preprocessor", rf_preprocessor), ("classifier",RandomForestClassifier(n_estimators = 50,
                                                                                                       max_depth=10,
                                                                                                       min_samples_leaf=10,
                                                                                                       
                                                                                                       class_weight= "balanced",
                                                                                                       n_jobs= -1,
                                                                                                       random_state= 42))])


In [45]:
# We fit RF model
rf_pipeline.fit(X_train, y_train);

Next we evaluate the performance of the RF model on the training set

In [47]:
y_train_pred = rf_pipeline.predict(X_train)

We evaluate the performance of the RF model

In [49]:
print(classification_report(y_train, y_train_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.89      0.86      0.87       162
   death_event       0.72      0.78      0.75        77

      accuracy                           0.83       239
     macro avg       0.81      0.82      0.81       239
  weighted avg       0.84      0.83      0.83       239



We evaluate the performance on the test set

In [51]:
y_test_pred = rf_pipeline.predict(X_test)
print(classification_report(y_test, y_test_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.84      0.76      0.79        41
   death_event       0.57      0.68      0.62        19

      accuracy                           0.73        60
     macro avg       0.70      0.72      0.71        60
  weighted avg       0.75      0.73      0.74        60



## Hyperparameter Tuning

We tune the parameters and use cross validation for possible model improvement

In [54]:
# Choice of hyperparameter

# several possible params combinations have been used. This choice seems better among others tested.
param_grid_rf = {"classifier__n_estimators":[3], 
                 "classifier__max_depth": [10],
                 "classifier__min_samples_leaf": [3,5,10,15],
                
                "classifier__class_weight": ["balanced"]}

In [55]:
# Cross validation step
rf_cv = StratifiedKFold(n_splits = 5, shuffle=True,random_state=42)

In [56]:
# We prioritize class 1 recall
score = make_scorer(recall_score, pos_label = 1)


In [57]:
# We search for the best parameters
grid_search_rf = GridSearchCV(estimator = rf_pipeline,
                                   param_grid = param_grid_rf,
                                   cv= rf_cv,
                                   scoring=score,
                                   refit=True,
                                   n_jobs=-1)

In [58]:
# We proceed to fit the model
grid_search_rf.fit(X_train, y_train);

In [59]:
# We check the best parameters
print("Best parameters:", grid_search_rf.best_params_)

Best parameters: {'classifier__class_weight': 'balanced', 'classifier__max_depth': 10, 'classifier__min_samples_leaf': 5, 'classifier__n_estimators': 3}


We choose the best model

In [61]:
best_RF_model = grid_search_rf.best_estimator_

We predict using the best model

In [63]:
y_train_pred = best_RF_model.predict(X_train)

We check the model performance on the training set

In [65]:
print(classification_report(y_train, y_train_pred, target_names=["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.87      0.79      0.83       162
   death_event       0.63      0.75      0.69        77

      accuracy                           0.78       239
     macro avg       0.75      0.77      0.76       239
  weighted avg       0.79      0.78      0.78       239



We check the performance on test set

In [67]:
y_test_pred = best_RF_model.predict(X_test)


In [68]:
print(classification_report(y_test, y_test_pred, target_names=["no_death_event","death_event"]))

                precision    recall  f1-score   support

no_death_event       0.84      0.78      0.81        41
   death_event       0.59      0.68      0.63        19

      accuracy                           0.75        60
     macro avg       0.72      0.73      0.72        60
  weighted avg       0.76      0.75      0.75        60



In [69]:
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

[[32  9]
 [ 6 13]]


Interpretation:

After hyperparameter tuning, for the clinically important death event class, the model had a recall of 0.68, meaning that it correctly identified 68% of patients who actually experienced a death event. In practical terms, the model detected approximately 13 out 19 death-event cases, but missed 6. And the death event precision was 0.59, meaning among patients the model predicted as having a death event, about 59% actually experienced a death event.  

# Gradient Boost Model

We proceed to fit gradient boosting model in our dataset

In [74]:
gb_preprocessor = ColumnTransformer(transformers = [("numeric", "passthrough", numeric_cols), ("cat", "passthrough", categorical_cols)], remainder ="drop")

In [75]:
gb_pipeline = Pipeline( steps = [("preprocessor", gb_preprocessor),("classifier", HistGradientBoostingClassifier(loss="log_loss",
                                                                                                                 learning_rate=0.05,
                                                                                                                max_iter=500,
                                                                                                                max_leaf_nodes=31,
                                                                                                                min_samples_leaf=20,
                                                                                                                l2_regularization=0.0,
                                                                                                                early_stopping=True,
                                                                                                                validation_fraction=0.15,
                                                                                                                n_iter_no_change =20,
                                                                                                                class_weight ="balanced",
                                                                                                                 random_state =42
                                                                                                                ))])

In [76]:
gb_pipeline.fit(X_train, y_train);

We evaluate performance on train set

In [78]:
y_train_pred = gb_pipeline.predict(X_train)

In [79]:
cm_gb = confusion_matrix(y_train, y_train_pred)
print(cm_gb)

[[136  26]
 [ 12  65]]


In [80]:
print(classification_report(y_train, y_train_pred, target_names = ["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.92      0.84      0.88       162
   death_event       0.71      0.84      0.77        77

      accuracy                           0.84       239
     macro avg       0.82      0.84      0.83       239
  weighted avg       0.85      0.84      0.84       239



We evaluate performance on test set

In [82]:
y_test_pred = gb_pipeline.predict(X_test)
cm_gb = confusion_matrix(y_test, y_test_pred)
print(cm_gb)

[[30 11]
 [ 6 13]]


In [83]:
print(classification_report(y_test, y_test_pred, target_names = ["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.83      0.73      0.78        41
   death_event       0.54      0.68      0.60        19

      accuracy                           0.72        60
     macro avg       0.69      0.71      0.69        60
  weighted avg       0.74      0.72      0.72        60



Interpretation

So far, on the test set, the Gradient Boosting model achieved a recall of 0.68 for the death event class. This means the model correctly identified 68% of patients who truly experienced a death event. The model also achieved a precision of 0.54 for the death event class. This means that, among the patients the model predicted as experiencing a death event, 54% actually experienced a death event. Overall, the model is moderately effective at detecting patients who experienced a death event, but its precision suggest that some patients are being misclassified as death event cases.

## Hyperparameter Tuning of the Gradient Boost Model

We apply cross-validation and tune the hyperparameters accordingly.

In [185]:
cv_gb = StratifiedKFold(n_splits=5, shuffle=True, random_state = 42)
scoring_gb = make_scorer(recall_score, pos_label = 1)
param_search_gb = {"classifier__learning_rate": loguniform(0.01, 0.2),
                  "classifier__max_iter": randint(200,1200),
                  "classifier__max_leaf_nodes": randint(15,80),
                  "classifier__max_depth": [None,2,3,4,5,6],
                  "classifier__l2_regularization": loguniform(1e-4,10),
                  "classifier__max_features": [0.6,0.8,1.0]} 

In [187]:
random_search_gb = RandomizedSearchCV(estimator = gb_pipeline,
                                     param_distributions = param_search_gb,
                                     n_iter = 30,
                                     cv = cv_gb,
                                     scoring= scoring_gb,
                                     refit = True,
                                     random_state = 42,
                                     n_jobs = -1
                                     
                                     )

In [207]:
random_search_gb.fit(X_train, y_train);

We view the best parameters

In [192]:
print(f"Best parameters: {random_search_gb.best_params_}")

Best parameters: {'classifier__l2_regularization': np.float64(0.1796562642379064), 'classifier__learning_rate': np.float64(0.011841130198891353), 'classifier__max_depth': 4, 'classifier__max_features': 0.8, 'classifier__max_iter': 1005, 'classifier__max_leaf_nodes': 16}


We choose and fit the best model

In [209]:
best_model_gb = random_search_gb.best_estimator_
best_model_gb.fit(X_train, y_train);

We evaluate the best model on the train set

In [211]:
y_train_pred = best_model_gb.predict(X_train)
print(classification_report(y_train, y_train_pred, target_names = ["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.91      0.83      0.87       162
   death_event       0.70      0.82      0.75        77

      accuracy                           0.83       239
     macro avg       0.80      0.83      0.81       239
  weighted avg       0.84      0.83      0.83       239



We evaluate the best gradient model on test set

In [213]:
y_test_pred = best_model_gb.predict(X_test)
print(classification_report(y_test, y_test_pred, target_names = ["no_death_event", "death_event"]))

                precision    recall  f1-score   support

no_death_event       0.83      0.73      0.78        41
   death_event       0.54      0.68      0.60        19

      accuracy                           0.72        60
     macro avg       0.69      0.71      0.69        60
  weighted avg       0.74      0.72      0.72        60



Next we will tune the hyperparameters around the best parameters.